# Notebook 23 — A Mini Evaluation Harness
**AI Models and Agents for Business · Week 12 · Session 23**

A leaderboard tells you which model is good at *someone else's* tasks. To know whether an assistant works for **your** use, you need your own test. Today you build a small one: a test set, a scoring rule, and an **AI judge**, and then you test whether the judge can be trusted.

### What you will do
1. Score twelve candidate answers with a **simple rule** and compare it with **human judgment**.
2. Score them with an **LLM judge** and measure how often it agrees with the humans.
3. Test the judge for **position bias** (does it prefer whichever answer comes first?) and **verbosity bias** (does it prefer the longer answer?).
4. Run a real assistant through the test and report a pass rate.
5. Write your own test cases.

### The scenario
A fictional retailer, **Summit Gear Co.**, has a returns policy. Its support assistant must answer questions from that policy only. Your job is to evaluate it.

### One-time setup: store your key in Colab Secrets
1. Click the **🔑 key icon** in the left sidebar of Colab.
2. Click **Add new secret**. Name: `GROQ_API_KEY`. Value: your key (it starts with `gsk_`).
3. Turn on **Notebook access** for this notebook.

Colab will remember the key for all your notebooks, so you never paste it into a cell. If you skip this, the setup cell will simply ask you to paste the key each time (the box hides what you type).

In [ ]:
# Run this cell first. Do not modify.
!pip install groq --quiet

import time, json, textwrap
from groq import Groq
import groq

def _get_key():
    """Look for the key in Colab Secrets first; otherwise ask for it (hidden as you type)."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

API_KEY = _get_key()
client = Groq(api_key=API_KEY)
MODEL = "openai/gpt-oss-20b"         # if Groq retires this name, change it here only

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def _retry_after(err, default=15):
    """Seconds the free tier asks us to wait (from the error's retry-after header), capped at 60."""
    try:
        return min(60.0, float(err.response.headers.get("retry-after", default)))
    except Exception:
        return float(default)

def ask(prompt, system=None, temperature=0.0, max_tokens=300, messages=None, model=None):
    """Send one request to the model and return the full response object.
    Waits and retries if the free tier says 'slow down'."""
    if messages is None:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    for attempt in range(6):
        try:
            return client.chat.completions.create(model=model or MODEL, messages=messages,
                                                  temperature=temperature, max_tokens=max_tokens)
        except groq.RateLimitError as err:
            wait = _retry_after(err)
            print(f"(rate limit hit - waiting {wait:.0f} seconds, then trying again)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited. Wait a minute and re-run this cell.")

def say(prompt, **kw):
    """Same as ask(), but returns just the text of the answer."""
    return (ask(prompt, **kw).choices[0].message.content or "").strip()

print("✅ Setup complete. Model:", MODEL)

## Part A — The policy and the test set

In [ ]:
POLICY = """Summit Gear Co. Returns Policy
1. Items may be returned within 45 days of delivery for a full refund if unused and in original packaging.
2. Opened electronics (headlamps, GPS watches) have a shorter window: 15 days from delivery.
3. After the standard window, items can be returned for store credit up to 90 days after delivery.
4. Return shipping is free for defective items. For all other returns, a flat $7.95 fee is deducted from the refund.
5. Items priced over $200 require manager approval before a return is accepted.
6. Items marked "Final Sale" cannot be returned.
7. Refunds are issued to the original payment method within 5 to 7 business days.
"""

# Each case: question, reference answer (what a correct answer must convey),
#            candidate answer (what an assistant said), human verdict (pass/fail), keywords for the rule-based scorer
CASES = [
    {"q": "How long do I have to return an unused jacket?", "ref": "45 days from delivery.",
     "cand": "You can return an unused jacket within 45 days of delivery for a full refund.", "human": "pass", "kw": ["45 days"]},
    {"q": "How long do I have to return an opened headlamp?", "ref": "15 days from delivery.",
     "cand": "Opened electronics like headlamps can be returned within 45 days.", "human": "fail", "kw": ["15 days"]},
    {"q": "What does it cost to return a non-defective item?", "ref": "A flat $7.95 fee deducted from the refund.",
     "cand": "Return shipping costs a flat $7.95, which is deducted from your refund.", "human": "pass", "kw": ["7.95"]},
    {"q": "Who pays return shipping on a defective item?", "ref": "Summit Gear pays; it is free.",
     "cand": "Return shipping is free for defective items.", "human": "pass", "kw": ["free"]},
    {"q": "How long does a refund take?", "ref": "5 to 7 business days, to the original payment method.",
     "cand": "Refunds go back to your original payment method in about a week.", "human": "pass", "kw": ["5-7", "5 to 7"]},
    {"q": "Can I return a Final Sale item?", "ref": "No, Final Sale items cannot be returned.",
     "cand": "Final Sale items can be returned for store credit within 90 days.", "human": "fail", "kw": ["cannot", "can't", "not be returned"]},
    {"q": "I want to return a $250 tent. Anything special?", "ref": "Manager approval is required for items over $200.",
     "cand": "Items over $200 need manager approval, so your tent return must be approved first.", "human": "pass", "kw": ["manager approval"]},
    {"q": "What if it has been 60 days?", "ref": "Store credit, available up to 90 days after delivery.",
     "cand": "At 60 days you can still return it for store credit, up to 90 days after delivery.", "human": "pass", "kw": ["90 days"]},
    {"q": "Do you ship internationally?", "ref": "The policy does not say (should not invent an answer).",
     "cand": "Yes, we ship to Canada and Mexico.", "human": "fail", "kw": ["does not say", "not say", "not mention", "don't know"]},
    {"q": "How long do I have to return unused hiking boots?", "ref": "45 days from delivery.",
     "cand": "I'm sorry, I don't have information about return windows.", "human": "fail", "kw": ["45 days"]},
    {"q": "Can I get my refund in cash?", "ref": "Refunds go to the original payment method.",
     "cand": "Refunds are issued to your original payment method, so cash refunds are not offered.", "human": "pass", "kw": ["original payment method"]},
    {"q": "How long do I have for an opened GPS watch?", "ref": "15 days from delivery.",
     "cand": "You have 15 days for opened electronics like GPS watches, though most customers find 45 days is typical.", "human": "fail", "kw": ["15 days"]},
]
print(len(CASES), "cases;", sum(c["human"] == "pass" for c in CASES), "pass and", sum(c["human"] == "fail" for c in CASES), "fail according to the human reviewer")

Read through the cases. The "human" column is the judgment of a careful reviewer. **Do you agree with every one?** Disagreement among reviewers is itself a finding: write down any case you would label differently.

## Part B — Scorer 1: a simple rule

The rule: **pass if the answer contains a required keyword.** It is fast, free, and completely predictable. How often does it agree with the human?

In [ ]:
def rule_score(case):
    text = case["cand"].lower()
    return "pass" if any(k.lower() in text for k in case["kw"]) else "fail"

import pandas as pd
rows = []
for i, c in enumerate(CASES, 1):
    rows.append({"#": i, "human": c["human"], "rule": rule_score(c), "answer": c["cand"][:60]})
df_rule = pd.DataFrame(rows)
df_rule["agree"] = df_rule["human"] == df_rule["rule"]
print(df_rule.to_string(index=False))
print()
print(f"Rule agrees with the human on {df_rule['agree'].sum()} of {len(df_rule)} cases ({df_rule['agree'].mean():.0%}).")

💬 **Discuss:** Look at the disagreements. One is a **false alarm** (rule says fail, human says pass) and one is a **miss** (rule says pass, human says fail). Which is more dangerous for a business? What kinds of correct answers does a keyword rule miss? What kinds of wrong answers does it let through?

## Part C — Scorer 2: an AI judge

A model reads the question, the reference answer and the candidate answer and returns a verdict. This handles paraphrases, and can catch an answer that contains the right number **plus** a wrong claim, but it is itself a model that can be wrong.

In [ ]:
import re

JUDGE_SYSTEM = ("You are a strict grader of customer-support answers. You are given a question, a reference answer, and a candidate answer. "
                "The candidate PASSES only if everything it states is consistent with the reference and it conveys the reference's key fact. "
                "It FAILS if it contradicts the reference, adds a wrong claim, invents information, or refuses when the reference has an answer. "
                'Reply with JSON only: {"verdict": "pass" or "fail", "reason": "<one short sentence>"}')

def judge(case, candidate=None):
    cand = candidate if candidate is not None else case["cand"]
    prompt = f"Question: {case['q']}\nReference answer: {case['ref']}\nCandidate answer: {cand}"
    raw = say(prompt, system=JUDGE_SYSTEM, temperature=0.0, max_tokens=80)
    m = re.search(r"\{.*?\}", raw, re.S)
    try:
        data = json.loads(m.group(0)) if m else {}
    except Exception:
        data = {}
    verdict = str(data.get("verdict", "")).lower()
    if verdict not in ("pass", "fail"):
        verdict = "pass" if re.search(r"\bpass\b", raw.lower()) and "fail" not in raw.lower() else "fail"
    return verdict, data.get("reason", raw[:80])

judge_rows = []
for i, c in enumerate(CASES, 1):
    v, why = judge(c)
    judge_rows.append({"#": i, "human": c["human"], "judge": v, "reason": why[:70]})
df_judge = pd.DataFrame(judge_rows)
df_judge["agree"] = df_judge["human"] == df_judge["judge"]
print(df_judge.to_string(index=False))
print()
print(f"Judge agrees with the human on {df_judge['agree'].sum()} of {len(df_judge)} cases ({df_judge['agree'].mean():.0%}).")

In [ ]:
# A confusion table: rows = what the human said, columns = what the judge said
print(pd.crosstab(df_judge["human"], df_judge["judge"], margins=True))
print()
print("Reminder: with 12 cases, ONE case changes the agreement by about 8 points.")
print("Is the judge better than the keyword rule on this sample? How sure can you be?")

## Part D — Can the judge be fooled?

Two known weaknesses of AI judges:

- **Position bias:** when comparing two answers, it prefers the one shown first (or second), regardless of quality.
- **Verbosity bias:** it prefers the longer, more polished answer, even if both say the same thing.

### Test 1: position bias
Pairs where one answer is clearly correct and one clearly wrong. Ask "which is better?" twice, swapping the order. An unbiased judge picks the **same answer** both times.

In [ ]:
PAIRS = [
    ("How long do I have to return an unused jacket?", "45 days from delivery.", "30 days from delivery."),
    ("What does it cost to return a non-defective item?", "A flat $7.95 fee, deducted from the refund.", "Returns are always free."),
    ("Can I return a Final Sale item?", "No. Final Sale items cannot be returned.", "Yes, within 45 days."),
    ("How long does a refund take?", "5 to 7 business days.", "Within 24 hours."),
]
PAIR_SYSTEM = "You compare two answers to a customer question using the Summit Gear returns policy. Reply with only the letter of the better answer: A or B."
POLICY_NOTE = "\n\nPolicy:\n" + POLICY

def pick(q, a, b):
    out = say(f"Question: {q}\n\nAnswer A: {a}\n\nAnswer B: {b}{POLICY_NOTE}", system=PAIR_SYSTEM, temperature=0.0, max_tokens=5).strip().upper()
    return "A" if out.startswith("A") else "B" if out.startswith("B") else "?"

consistent = 0
for q, good, bad in PAIRS:
    first  = pick(q, good, bad)     # correct answer in slot A
    second = pick(q, bad, good)     # correct answer in slot B
    picked_good_1 = first == "A"
    picked_good_2 = second == "B"
    ok = picked_good_1 and picked_good_2
    consistent += ok
    print(f"{q[:48]:50s} correct-first picks correct: {picked_good_1} | correct-second picks correct: {picked_good_2}  {'✅' if ok else '⚠️ position effect'}")
print()
print(f"Judge picked the correct answer in BOTH orders for {consistent} of {len(PAIRS)} pairs.")

### Test 2: verbosity bias
Now both answers are **correct and say the same thing**: one short, one long and padded. With no bias, a judge should pick each about half the time (or say they are equal). Run it.

In [ ]:
SHORT_LONG = [
    ("How long do I have to return an unused jacket?", "45 days from delivery.",
     "Great question, and thank you so much for shopping with us! We truly value you as a customer. You have 45 days from the date of delivery to return an unused jacket, so please take your time and let us know if there is anything else we can possibly help you with today."),
    ("Who pays return shipping on a defective item?", "We do. It is free.",
     "We completely understand how frustrating it is to receive a defective item, and we apologize for any inconvenience. Rest assured that return shipping on defective items is entirely free of charge, so there is nothing at all for you to worry about."),
    ("How long does a refund take?", "5 to 7 business days.",
     "We know you are eager to see your money, and we work hard to process every refund as quickly as we possibly can. You can expect your refund to arrive in 5 to 7 business days, issued to your original payment method."),
    ("Can I return a Final Sale item?", "No.",
     "Thank you for reaching out about this! Unfortunately, items that are marked as Final Sale are not eligible for return, and we sincerely appreciate your understanding in this matter."),
]
picked_long = 0
for q, short, long_ in SHORT_LONG:
    r1 = pick(q, short, long_)     # long is B
    r2 = pick(q, long_, short)     # long is A
    long_wins = (r1 == "B") + (r2 == "A")
    picked_long += long_wins
    print(f"{q[:46]:48s} picked the LONG answer {long_wins} of 2 times")
print()
print(f"Long answer chosen {picked_long} of {2*len(SHORT_LONG)} comparisons (50% would mean no length bias).")

💬 **Discuss:** What would happen if you used this judge to *rank* assistants, when one of them writes long, friendly answers? How could you reduce these biases? (Ideas: randomize the order and average; judge one answer at a time against a reference; give a rubric; mix in human labels.)

## Part E — Evaluate a real assistant

Now put an assistant through the test. The assistant gets the policy and a question. The judge compares each answer with the reference. Then **change the assistant's instructions** and see whether the pass rate moves.

In [ ]:
# ✏️ CHANGE ME — try different instructions for the assistant
ASSISTANT_SYSTEM = "You are a customer support assistant for Summit Gear Co. Answer using ONLY the returns policy below. If the policy does not say, say you don't know.\n\n" + POLICY

def run_assistant(system):
    verdicts = []
    for c in CASES:
        answer = say(c["q"], system=system, temperature=0.0, max_tokens=120)
        v, why = judge(c, candidate=answer)
        verdicts.append((c["q"], answer, v))
    return verdicts

run1 = run_assistant(ASSISTANT_SYSTEM)
rate = sum(v == "pass" for _, _, v in run1) / len(run1)
print(f"Pass rate with your instructions: {rate:.0%}\n")
for q, a, v in run1:
    print(("✅" if v == "pass" else "❌"), q)
    if v == "fail":
        print("     assistant said:", a[:160].replace("\n", " "))

In [ ]:
# For comparison: the same assistant WITHOUT the policy in its prompt
run0 = run_assistant("You are a customer support assistant for Summit Gear Co.")
rate0 = sum(v == "pass" for _, _, v in run0) / len(run0)
print(f"Pass rate without the policy text: {rate0:.0%}   (with it: {rate:.0%})")
print("A test that shows this gap is a test that can detect whether grounding is working.")

## Part F — ✏️ Write three test cases of your own
A test set grows from failures. Add three cases that you think will trip up the assistant (a question the policy answers in two places, one with a trap, and one it does not cover). Then re-run Part E.

In [ ]:
# ✏️ CHANGE ME — add your own cases
MY_CASES = [
    # {"q": "...", "ref": "...", "cand": "(leave blank; the assistant will answer)", "human": "pass", "kw": ["..."]},
]

CASES_BACKUP = list(CASES)
CASES = CASES_BACKUP + MY_CASES
print("Test set now has", len(CASES), "cases. Re-run Part E to see the new pass rate.")

## Carry this into Project 3
For the audit you will write a test set and a scoring method for **another team's** assistant. Decide *before* you start: what is the reference answer for each probe, what counts as a pass, and will you score by rule, by judge, or by hand? Whatever you choose, **check it against your own hand-labels** as you did here and report the agreement.

---
## ✅ Notebook 23 complete — Reflection (submit with your notebook)
1. What was the rule's agreement with the human? The judge's? What kind of error did each make?
2. Describe the position or verbosity result you observed. What could that do to a leaderboard?
3. What was the assistant's pass rate with the policy and without it? What does the gap tell you?
4. Why is 12 test cases not enough to be sure about a score? How many would you want, and why?

*Next session: Adversarial Behavior and Guardrails. Project 3 probe logs are due.*